In [21]:
import os, time, warnings
warnings.filterwarnings("ignore")


import logfire 
from dotenv import load_dotenv

load_dotenv(override=True)


# Verify keys
print("LOGFIRE_TOKEN  :", "✅" if os.getenv("LOGFIRE_TOKEN")  else "❌  missing")
print("GROQ_API_KEY   :", "✅" if os.getenv("GROQ_API_KEY")   else "❌  missing")
print("GEMINI_API_KEY :", "✅" if os.getenv("GEMINI_API_KEY") else "❌  missing")

LOGFIRE_TOKEN  : ✅
GROQ_API_KEY   : ✅
GEMINI_API_KEY : ✅


---
##  Part 1 — Why Logfire & First Traces

The problem with `print()` in production:

| `print()` | `logfire` |
|-----------|-----------|
| Plain string, unsearchable | Structured key-value fields, fully searchable |
| No timestamp or duration | Automatic timestamps, span duration |
| Lost in terminal noise | Real-time dashboard with filters and queries |
| Nothing in production | Persisted traces, alerting, cost analytics |

Logfire is built on **OpenTelemetry** — the industry standard. Every trace you write here is portable.

In [22]:
import logfire

logfire.configure()

logfire.info("Hello, {place}!", place="udemy")

Logfire project URL: https://logfire-us.pydantic.dev/jayanththalla/udemy

16:25:38.133 Hello, udemy!


In [23]:
logfire.info("notebook started",name="notebook", tags={"module":"pydantic_logfire"})

16:26:44.881 notebook started


In [24]:
logfire.configure(
    token=os.getenv("LOGFIRE_TOKEN"),
    service_name="llm-observability-course"
)

Logfire project URL: https://logfire-us.pydantic.dev/jayanththalla/udemy

In [25]:
logfire.info("notebook_started",
            part="PART 1 - BASICS",
            instructer = "Divesh",
            tool = "Pydantic Logfire"
            )

16:27:33.215 notebook_started


In [26]:
with logfire.span("data_processing_simulation", dataset="llm_course", rows=1000):
    logfire.info("step_started", step=1, action="loading data")
    time.sleep(0.3)

    logfire.info("step_started", step=2, action="transforming", columns=12)
    time.sleep(0.2)

    logfire.info("step_started", step=3, action="saving results", output="/tmp/out.csv")

16:27:41.541 data_processing_simulation
16:27:41.545   step_started
16:27:41.847   step_started
16:27:42.052   step_started


### Experiment 2 — Structured Logging with Pydantic Models

The *"Pydantic"* in Pydantic Logfire: when you log a Pydantic model, Logfire **expands every field** into a searchable attribute automatically.

In a real LLM app you log request/response objects hundreds of times per minute. With string logging you get `"{'user_id': 'alice', ...}"` — unsearchable. With Logfire you get filterable columns.

In [28]:
from pydantic import BaseModel
from typing import Optional

# MOCK DATA nOT REAL DATA

class LLMRequest(BaseModel):
    user_id: str
    session_id: str
    query: str
    model: str
    temperature: float = 0.7
    max_tokens: Optional[int] = None


class LLMResponse(BaseModel):
    answer: str
    input_tokens: int
    output_tokens: int
    latency_ms: float
    model_used: str    

In [ ]:
# ── Simulate logging a real LLM request/response ──────────────────────────
request = LLMRequest(
    user_id="priya",
    session_id="sess_abc123",
    query="What is Retrieval-Augmented Generation?",
    model="openai/gpt-oss-120b",
    max_tokens=500
)

with logfire.span("llm_CALL",
                  user_id = request.user_id,
                  session_id = request.session_id,
                  model_used = request.model):
    logfire.info("request_received" , **request.model_dump())
    
    time.sleep(0.1)

    response = LLMResponse(
        answer="RAG is a technique that retrieves relevant documents...",
        input_tokens=18,
        output_tokens=120,
        latency_ms=342.5,
        model_used="openai/gpt-oss-120b"
    )
    logfire.info("response_sent", **response.model_dump())


print(response)


16:33:56.728 llm_CALL
16:33:56.739   request_received
16:33:56.842   response_sent
answer='RAG is a technique that retrieves relevant documents...' input_tokens=18 output_tokens=120 latency_ms=342.5 model_used='llama-3.3-70b-versatile'


---
##  Part 2 — Auto-Instrumentation of LLM Calls

In Part 1 you wrote spans manually. Now let Logfire do it **automatically**.

`logfire.instrument_openai()` patches the OpenAI Python SDK.
Since **Groq** and **Gemini** both expose an OpenAI-compatible REST API, the same single line instruments all of them.

Every LLM call then automatically records:
- Model name (`llama-3.3-70b-versatile`, `gemini-2.5-flash-lite`, …)
- Input + output token counts
- Wall-clock latency
- Full prompt text and full response text

You write **zero** extra logging code. It just appears in the dashboard.

###  Experiment 3 — Instrument Groq (llama-3.3-70b)

We use `ChatOpenAI` from LangChain pointed at Groq's OpenAI-compatible endpoint.
`logfire.instrument_openai()` patches the underlying SDK — Groq calls appear in traces automatically.

```
Your code  →  ChatOpenAI(base_url="https://api.groq.com/openai/v1")
                          ↓
              [logfire.instrument_openai() intercepts here]
                          ↓
              Groq API  →  response
```

In [31]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

logfire.instrument_openai()

llm_groq=ChatOpenAI(model="openai/gpt-oss-120b",base_url="https://api.groq.com/openai/v1", api_key=os.getenv("GROQ_API_KEY"), temperature=0.7, max_tokens=500)

print("Calling GRoq LLM...")
response = llm_groq.invoke([HumanMessage(content="What is Retrieval-Augmented Generation?")])
print(response.content)

Calling GRoq LLM...
16:39:33.815 Chat Completion with 'openai/gpt-oss-120b' [LLM]
**Retrieval‑Augmented Generation (RAG)** is a hybrid AI architecture that couples a **retriever** (a search‑or‑lookup component) with a **generator** (usually a large language model, LLM). The system first pulls relevant pieces of external information—documents, passages, tables, code snippets, etc.—and then feeds those retrieved texts into the generator, which uses them as “grounding” to produce a final answer or piece of content.

Below is a step‑by‑step view of how a typical RAG pipeline works, followed by its motivations, variants, and practical tips.

---

## 1. Core Workflow

| Stage | What Happens | Typical Technologies |
|-------|--------------|----------------------|
| **1️⃣ Query Encoding** | The user’s prompt is turned into a dense vector (embedding) that captures its semantic meaning. | Sentence‑Transformers, OpenAI embeddings, Cohere embed, etc. |
| **2️⃣ Retrieval** | The embedding is matche

###  Experiment 4 — Instrument Gemini (gemini-2.5-flash-lite)

Gemini now exposes an OpenAI-compatible endpoint too:
`https://generativelanguage.googleapis.com/v1beta/openai/`

Same `ChatOpenAI` setup — just a different `base_url` and model name.
`logfire.instrument_openai()` was already called — no need to call it again.

After this cell, your dashboard will show traces from **two different model names** in the same project.

In [32]:
# Gemini's OpenAI-compatible endpoint (no extra setup needed)
llm_gemini = ChatOpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.getenv("GEMINI_API_KEY"),
    model="gemini-2.5-flash-lite",
    temperature=0.3
)


print("Calling Gemini (gemini-2.5-flash-lite)…")
try:
    response = llm_gemini.invoke([
        HumanMessage(content="Explain what an observability 'trace' is, in exactly 2 sentences.")
    ])
    print(f"\n🔵 Gemini Response:\n{response.content}")
except Exception as e:
    print(f"⚠️  Gemini call failed: {e}")
    print("    Check your GEMINI_API_KEY in .env")

Calling Gemini (gemini-2.5-flash-lite)…
16:41:26.055 Chat Completion with 'gemini-2.5-flash-lite' [LLM]

🔵 Gemini Response:
An observability trace is a record of the end-to-end journey of a request as it travels through a distributed system. It captures the sequence of operations, their timings, and any errors encountered, providing a holistic view of request flow.


###  Experiment 5 — Side-by-Side: Groq vs Gemini in One Trace Waterfall

Wrap both calls in a **parent span** — the dashboard shows them as a waterfall:

```
model_comparison  ←── parent span (total wall time)
  ├── groq_call   ←── child span  (latency: Xms, tokens: N)
  └── gemini_call ←── child span  (latency: Xms, tokens: N)
```

This is exactly how you run **A/B model tests in production** — one trace per comparison, all searchable.

In [ ]:
query = "What is the difference between RAG and fine-tuning? Give 3 bullet points."

with logfire.span("model_comparison", query=query, num_models=2):

    # ── Groq ─────────────────────────────────────────────────────────────
    with logfire.span("groq_call", model="openai/gpt-oss-120b", provider="groq"):
        t0 = time.time()
        r_groq = llm_groq.invoke([HumanMessage(content=query)])
        groq_ms = round((time.time() - t0) * 1000, 1)
        logfire.info("groq_done", latency_ms=groq_ms, answer_len=len(r_groq.content))

    # ── Gemini ────────────────────────────────────────────────────────────
    with logfire.span("gemini_call", model="gemini-2.5-flash-lite", provider="google"):
        t0 = time.time()
        try:
            r_gemini = llm_gemini.invoke([HumanMessage(content=query)])
            gemini_ms = round((time.time() - t0) * 1000, 1)
            logfire.info("gemini_done", latency_ms=gemini_ms, answer_len=len(r_gemini.content))
            gemini_answer = r_gemini.content
        except Exception as e:
            logfire.warning("gemini_failed", error=str(e))
            gemini_ms = 0
            gemini_answer = f"[Error: {e}]"

# ── Print results ─────────────────────────────────────────────────────────
print(f"🟢 Groq ({groq_ms}ms):\n{r_groq.content}")
print(f"\n🔵 Gemini ({gemini_ms}ms):\n{gemini_answer}")

16:41:55.125 model_comparison
16:41:55.127   groq_call
16:41:55.130     Chat Completion with 'openai/gpt-oss-120b' [LLM]
16:41:56.177     groq_done
16:41:56.180   gemini_call
16:41:56.187     Chat Completion with 'gemini-2.5-flash-lite' [LLM]
16:41:58.115     gemini_done
🟢 Groq (1048.9ms):
- **How knowledge is incorporated**  
  - **RAG:** The model pulls relevant information from an external datastore at inference time (e.g., via a vector search) and conditions its generation on those retrieved passages.  
  - **Fine‑tuning:** The knowledge is baked directly into the model’s parameters during a training step; no external lookup is needed at runtime.

- **Flexibility & update speed**  
  - **RAG:** Adding, correcting, or deleting content only requires updating the underlying index or knowledge base—changes take effect instantly without retraining the model.  
  - **Fine‑tuning:** To reflect new information you must run another fine‑tuning pass (or full retraining), which can be time‑co

---
##  Part 3 — RAG Pipeline Tracing

A RAG pipeline has **3 stages**, each taking time and each capable of failing:

```
User Query
    │
    ▼
[Embed Query]        ← Gemini text-embedding API call
    │
    ▼
[Retrieve Docs]      ← FAISS similarity search (in-memory)
    │
    ▼
[Generate Answer]    ← LLM API call (Groq llama-3.3-70b)
    │
    ▼
Response
```

Without observability: *"It's slow"* — but which stage? The embedding? The retrieval? The LLM?
With Logfire: you see **exactly** which stage takes how long and what data flows through each one.

**Stack:**
- Embeddings: `gemini-embedding-2-preview` via Gemini API — no local model, reuses your existing key
- Vector store: `FAISS` — in-memory, no server needed
- LLM: Groq `openai/gpt-oss-120b`

###  Experiment 6 — Traced RAG Pipeline

Load documents from `documents.json`, embed them with Gemini, build a FAISS index, and run a traced RAG pipeline.

Tracing shows the two things that matter:
- **Which documents were retrieved** (relevance check without opening the code)
- **LLM call** — model, tokens, latency, full prompt+response (auto-captured by `instrument_openai`)

In [36]:
import json
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document


# ── Load knowledge base ────────────────────────────────────────────────────
with open("documents.json") as f:
    raw_docs = json.load(f)
    
# CONVERT THE DOCS IN LANGCHAIN COMPATIPBLE DOCUMENTS

DOCS = [
    Document(page_content=d["content"], metadata={"topic": d["topic"], "source": d["source"]})
    for d in raw_docs
]
print(f"Loaded {len(DOCS)} documents: {[d.metadata['topic'] for d in DOCS]}")


Loaded 6 documents: ['RAG', 'Guardrails', 'Gateway', 'Observability', 'Evals', 'Fine-tuning']


In [37]:
# ── Embeddings + FAISS index ───────────────────────────────────────────────
embeddings  = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

vectorstore = FAISS.from_documents(DOCS, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 2})
print("✅  FAISS index ready")



✅  FAISS index ready


In [38]:
# ── RAG with tracing ───────────────────────────────────────────────────────
def rag(question: str, user_id: str = "anonymous") -> str:
    with logfire.span("rag_pipeline", question=question, user_id=user_id):
        docs = retriever.invoke(question)   # search similar vectors 
        logfire.info("docs_retrieved",
                     topics=[d.metadata["topic"] for d in docs],
                     num_docs=len(docs))
        context = "\n\n".join(
            f"[{d.metadata['topic']}] {d.page_content}" for d in docs
        )
        prompt = (
            f"Answer the question based only on the context below.\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {question}\n\nAnswer concisely:"
        )
        return llm_groq.invoke(prompt).content


In [40]:
answer = rag("How does a RAG reduce hallucination", user_id="priya")
print(f"\n🟢 RAG Answer:\n{answer}")

16:50:41.768 rag_pipeline
16:50:42.380   docs_retrieved
16:50:42.385   Chat Completion with 'openai/gpt-oss-120b' [LLM]

🟢 RAG Answer:
RAG first pulls relevant documents from a knowledge base and feeds those retrieved passages to the LLM along with the user’s query. Because the model’s response is grounded in actual retrieved content—rather than relying solely on its internal memorized knowledge—it can cite that source and is less likely to fabricate (hallucinate) information. This retrieval‑grounding keeps answers tied to verifiable data and updates automatically as the knowledge base changes.


## React Agent

###  Experiment 7 — LangGraph ReAct Agent with Retriever as a Tool

A fixed RAG chain always retrieves. An agent decides *when* to retrieve.

The retriever is a plain Python function decorated with `@tool` — the agent calls it only when the question needs knowledge-base lookup.

```
agent_run  (logfire span)
  ├── LLM decides: call tool or answer directly?
  ├── search_knowledge_base()  ← only if needed
  └── LLM generates final answer
```

In [41]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage, AIMessage
# ── Retriever as a plain Python tool ──────────────────────────────────────
@tool
def search_knowledge_base(query: str) -> str:
    """Search the knowledge base for LLM production topics: RAG, guardrails,
    gateways, observability, evaluations, and fine-tuning."""
    docs = vectorstore.similarity_search(query, k=2)
    return "\n\n".join(
        f"[{d.metadata['topic']}] {d.page_content}" for d in docs
    )

agent = create_agent(
    model=llm_groq,
    tools=[search_knowledge_base],
    system_prompt=(
        "You are a helpful assistant. Use search_knowledge_base for any question "
        "about LLM production topics. Answer directly for general knowledge questions."
    )
)

In [42]:
# ── Traced agent runner ────────────────────────────────────────────────────
def run_agent(question: str, user_id: str = "anonymous"):
    with logfire.span("agent_run", question=question, user_id=user_id):
        result = agent.invoke({"messages": [HumanMessage(content=question)]})

        # Last message may be a ToolMessage or an AIMessage with empty content
        # — iterate backwards to find the last AIMessage with actual text
        last_ai = next(
            (m for m in reversed(result["messages"]) if isinstance(m, AIMessage)),
            None
        )
        answer    = last_ai.text if last_ai else ""
        used_tool = any(isinstance(m, ToolMessage) for m in result["messages"])

        logfire.info("agent_done", used_tool=used_tool, answer_length=len(answer))
        return answer, used_tool


In [43]:
# ── Test ───────────────────────────────────────────────────────────────────
queries = [
    ("What is LLM observability and which tools provide it?", "priya"),
    ("How do LLM guardrails work?",  "bhavesh"),
    ("What is the capital of France?", "kunal"),
]

In [44]:
for q, uid in queries:
    print(f"{'='*55}")
    answer, used_tool = run_agent(q, user_id=uid)
    print(f"Q: {q}")
    print(f"Tool used: {used_tool}  {'← retrieved from KB' if used_tool else '← answered directly'}")
    print(f"A: {answer[:300]}")

16:55:08.464 agent_run
16:55:08.523   Chat Completion with 'openai/gpt-oss-120b' [LLM]
16:55:09.913   Chat Completion with 'openai/gpt-oss-120b' [LLM]
16:55:11.345   agent_done
Q: What is LLM observability and which tools provide it?
Tool used: True  ← retrieved from KB
A: **LLM observability** is the capability to monitor, trace, and debug language‑model‑driven applications once they are running in production.  
It goes beyond simple logging by giving you structured, end‑to‑end visibility into every step of the LLM pipeline, such as:

| Aspect | What it means for an 
16:55:11.348 agent_run
16:55:11.358   Chat Completion with 'openai/gpt-oss-120b' [LLM]
16:55:12.594   Chat Completion with 'openai/gpt-oss-120b' [LLM]
16:55:14.114   agent_done
Q: How do LLM guardrails work?
Tool used: True  ← retrieved from KB
A: **LLM guardrails** are safety‑and‑policy controls that sit between a user (or upstream system) and the language model. Instead of trusting the raw model to “just work,” you inte